[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/QuantLet/VaR_CEE_FM/blob/main/VaR_CEE_DataPipeline/VaR_CEE_DataPipeline.ipynb)

# VaR_CEE_DataPipeline

Computes log returns from raw price data and produces descriptive statistics including
mean, standard deviation, skewness, kurtosis, Jarque-Bera, ADF, and ARCH-LM tests
for all CEE market series.

**Published in:** *Can Foundation Models Manage Risk? Zero-Shot VaR and ES Forecasting for CEE Markets*

**Author:** Daniel Traian Pele

In [ ]:
# Install dependencies (Colab)
!pip install -q statsmodels

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from scipy import stats as sp_stats
from google.colab import files

%matplotlib inline

# ── Configuration ────────────────────────────────────────
MARKETS = {
    "Romania":  {"index": "BET",   "fx": "EURRON"},
    "Poland":   {"index": "WIG20", "fx": "EURPLN"},
    "Czechia":  {"index": "PX",    "fx": "EURCZK"},
    "Hungary":  {"index": "BUX",   "fx": "EURHUF"},
    "Bulgaria": {"index": "SOFIX", "fx": "USDBGN"},
}
START_DATE = "2007-01-01"
END_DATE = "2025-12-31"
OOS_START = "2018-01-01"

RAW_DIR = Path("data/raw")
PROC_DIR = Path("data/processed")
STATS_DIR = Path("data/stats")
FIG_DIR = Path("data/figures")
for d in [RAW_DIR, PROC_DIR, STATS_DIR, FIG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

## Upload Raw CSV Files

Upload the 10 CSV files: `BET.csv`, `WIG20.csv`, `PX.csv`, `BUX.csv`, `SOFIX.csv`,
`EURRON.csv`, `EURPLN.csv`, `EURCZK.csv`, `EURHUF.csv`, `USDBGN.csv`

Each CSV must have columns: `Date, Open, High, Low, Close, Volume` (or at least `Date, Close`).

In [ ]:
print("Upload the 10 raw CSV files:")
uploaded = files.upload()
# Move uploaded files to data/raw/
for fname, content in uploaded.items():
    dest = RAW_DIR / fname
    with open(dest, "wb") as f:
        f.write(content)
    print(f"  Saved {dest}")
print(f"\nUploaded {len(uploaded)} files")

In [ ]:
def load_raw(filename):
    """Load raw CSV, return Series of closing prices."""
    fpath = RAW_DIR / filename
    if not fpath.exists():
        print(f"  WARNING: {fpath} not found")
        return None
    df = pd.read_csv(fpath, index_col=0, parse_dates=True)
    if "Close" in df.columns:
        s = df["Close"].dropna()
    elif "close" in df.columns:
        s = df["close"].dropna()
    else:
        s = df.iloc[:, -1].dropna()
    s = s.sort_index()
    s = s[~s.index.duplicated(keep='first')]
    return s


def compute_log_returns(prices):
    """Compute log returns from price series."""
    return np.log(prices / prices.shift(1)).dropna()

In [ ]:
def adf_test(series):
    """Augmented Dickey-Fuller test."""
    from statsmodels.tsa.stattools import adfuller
    try:
        result = adfuller(series.dropna(), autolag='AIC')
        return result[0], result[1]
    except Exception:
        return np.nan, np.nan


def arch_lm_test(series, nlags=10):
    """Engle's ARCH-LM test."""
    try:
        from statsmodels.stats.diagnostic import het_arch
        result = het_arch(series.dropna(), nlags=nlags)
        return result[0], result[1]
    except Exception:
        return np.nan, np.nan


def compute_descriptive_stats(returns, name):
    """Compute descriptive statistics for a return series."""
    r = returns.dropna()
    n = len(r)
    jb_stat, jb_p = sp_stats.jarque_bera(r)
    adf_stat, adf_p = adf_test(r)
    arch_stat, arch_p = arch_lm_test(r)
    return {
        "Series": name, "N": n, "Mean": r.mean(), "Std": r.std(),
        "Min": r.min(), "Max": r.max(), "Skewness": r.skew(),
        "Kurtosis": r.kurtosis(), "JB_stat": jb_stat, "JB_p": jb_p,
        "ADF_stat": adf_stat, "ADF_p": adf_p,
        "ARCH_LM_stat": arch_stat, "ARCH_LM_p": arch_p,
    }

In [ ]:
# Process all markets and compute returns
print("=" * 60)
print("DATA PIPELINE -- RETURNS & DESCRIPTIVE STATS")
print("=" * 60)

all_stats = []
all_returns = {}

for country, info in MARKETS.items():
    print(f"\n--- {country} ---")
    idx_name = info['index']
    idx_prices = load_raw(f"{idx_name}.csv")
    fx_name = info['fx']
    fx_prices = load_raw(f"{fx_name}.csv")
    country_data = {}

    if idx_prices is not None:
        idx_ret = compute_log_returns(idx_prices)
        series_id = f"{idx_name}_ret"
        country_data[series_id] = idx_ret
        all_returns[series_id] = idx_ret
        print(f"  {series_id}: {len(idx_ret)} observations "
              f"({idx_ret.index[0].date()} to {idx_ret.index[-1].date()})")
        all_stats.append(compute_descriptive_stats(idx_ret, series_id))

    if fx_prices is not None:
        fx_ret = compute_log_returns(fx_prices)
        series_id = f"{fx_name}_ret"
        country_data[series_id] = fx_ret
        all_returns[series_id] = fx_ret
        print(f"  {series_id}: {len(fx_ret)} observations "
              f"({fx_ret.index[0].date()} to {fx_ret.index[-1].date()})")
        all_stats.append(compute_descriptive_stats(fx_ret, series_id))

    if country_data:
        df_country = pd.DataFrame(country_data)
        out_path = PROC_DIR / f"{country}_returns.parquet"
        df_country.to_parquet(out_path)
        print(f"  Saved {out_path}")

# Save descriptive statistics
if all_stats:
    df_stats = pd.DataFrame(all_stats)
    stats_path = STATS_DIR / "descriptive_stats.csv"
    df_stats.to_csv(stats_path, index=False, float_format="%.6f")
    print(f"\nDescriptive statistics saved to {stats_path}")

In [ ]:
# Display descriptive statistics table
if all_stats:
    display_cols = ["Series", "N", "Mean", "Std", "Skewness", "Kurtosis", "JB_p", "ADF_p"]
    print(df_stats[display_cols].to_string(index=False, float_format=lambda x: f"{x:.4f}"))

In [ ]:
# Plot log return series for all indices
index_series = {k: v for k, v in all_returns.items() if not k.startswith("EUR") and not k.startswith("USD")}

if index_series:
    fig, axes = plt.subplots(len(index_series), 1, figsize=(12, 3 * len(index_series)), sharex=False)
    if len(index_series) == 1:
        axes = [axes]
    for ax, (name, ret) in zip(axes, index_series.items()):
        ax.plot(ret.index, ret.values, linewidth=0.4, color='steelblue')
        ax.set_title(name, fontsize=10)
        ax.set_ylabel('Log return')
        ax.axhline(y=0, color='gray', linewidth=0.3)
    plt.tight_layout()
    plt.show()

In [ ]:
# Download processed data as zip
import zipfile, io

zip_buffer = io.BytesIO()
with zipfile.ZipFile(zip_buffer, "w", zipfile.ZIP_DEFLATED) as zf:
    for f in sorted(PROC_DIR.glob("*.parquet")):
        zf.write(f, f"processed/{f.name}")
    for f in sorted(STATS_DIR.glob("*.csv")):
        zf.write(f, f"stats/{f.name}")

zip_buffer.seek(0)
with open("processed_data.zip", "wb") as fout:
    fout.write(zip_buffer.read())
files.download("processed_data.zip")
print("Download started: processed_data.zip")